<a href="https://colab.research.google.com/github/tarkeshsingh/python-for-sensorimotor-control/blob/main/L10_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Static Optimization
## A Hands-On Tutorial for Movement Neuroscience Graduate Students

---

**Why this tutorial?**  
Your nervous system has more muscles than it has joints to move. Any movement can be produced by
infinitely many patterns of muscle force, and yet you produce *one*. Predicting which one requires
an extra assumption — a cost that the nervous system is presumed to minimize. That is static
optimization, and it is the tool behind Crowninshield & Brand (1981) and behind Lab 06.

By the end you will be able to set up a constrained minimization in SciPy, recognize when a problem
has no unique answer, and read the solution the optimizer gives you critically rather than
trusting it.

**Prerequisites:** Python Basics (L0), NumPy (L1), Matplotlib (L3).  
**Environment:** Google Colab (recommended) or Jupyter Notebook.  
**Time:** about 60–75 minutes.

---

## Table of Contents

**Part I — Unconstrained Optimization**
1. [What a Cost Function Is](#1)
2. [Minimizing in One Dimension](#2)
3. [Local vs Global: Why the Starting Point Matters](#3)
4. [Two Dimensions and Cost Landscapes](#4)

**Part II — Adding Constraints**
5. [Bounds: Keeping Variables Non-Negative](#5)
6. [Equality Constraints](#6)
7. [SLSQP: The Workhorse](#7)
8. [Supplying Gradients](#8)

**Part III — When There Is No Unique Answer**
9. [More Unknowns Than Equations](#9)
10. [Using a Cost to Choose](#10)
11. [What the Exponent Does](#11)

**Part IV — The Muscle Redundancy Problem**
12. [Six Muscles, Two Torques](#12)
13. [Reading the Solution Critically](#13)

**Part V — Practice**
14. [Exercises](#14)
15. [Summary & Further Reading](#15)

In [ ]:
# ---- Setup: import all libraries ----
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from matplotlib import cm

plt.rcParams.update({'figure.figsize': (9, 4), 'font.size': 11,
                     'axes.grid': True, 'grid.alpha': 0.3})
np.set_printoptions(precision=3, suppress=True)
print('Setup complete.')

---
# Part I — Unconstrained Optimization

## 1. What a Cost Function Is <a id='1'></a>

### Key concepts

An **optimization problem** has three parts:

| Part | Meaning | Example |
|---|---|---|
| **Decision variables** | what you are allowed to choose | muscle forces $F_1 \dots F_6$ |
| **Cost function** | a single number you want to make small | total muscle stress |
| **Constraints** | what any acceptable answer must satisfy | the forces must produce the required torque |

The cost function is written $J(x)$ and takes a vector of decision variables to one number.
**Optimization is the process of searching for the $x$ that makes $J$ smallest.**

A crucial point that catches people out: *the cost function is an assumption, not a measurement.*
Nothing in the mechanics tells you the nervous system minimizes anything. Choosing $J$ is a
modeling decision, and different choices give different predictions.

In [ ]:
# ---- Define a one-dimensional cost function ----
def J(x):
    """A cost function with one variable. Returns a single number."""
    return x * np.exp(-x**2) + (x**2) / 20

# Evaluate it at a few points
for x in [-2.0, -0.5, 0.0, 0.5, 2.0]:
    print(f'  J({x:+.1f}) = {J(x):+.4f}')

In [ ]:
# ---- Plot the cost landscape ----
x = np.linspace(-10, 10, 500)
plt.figure()
plt.plot(x, J(x), lw=2)
plt.xlabel('x'); plt.ylabel('J(x)'); plt.title('Cost landscape')
plt.axhline(0, color='k', lw=0.6)
plt.show()

**What you should see:** A curve with a deep dip just left of zero and a shallower dip near
$x \approx 2$. Both are places where the curve turns around — the function is lower there than
anywhere nearby. Hold on to the fact that there are **two** of them; Section 3 is about why that
matters.

---

## 2. Minimizing in One Dimension <a id='2'></a>

### Key function

`scipy.optimize.minimize(fun, x0)` — searches for the input that minimizes `fun`, starting the
search from `x0`.

It does not examine the whole landscape. It starts where you tell it and walks downhill until it
cannot go down any further.

In [ ]:
# ---- Run the optimizer from a starting guess of x = 0 ----
result = minimize(J, x0=0.0)
print(result)

In [ ]:
# ---- Pull out just the parts you usually need ----
print(f'  success : {result.success}')
print(f'  x*      : {result.x[0]:+.4f}')     # the minimizing input (always an array)
print(f'  J(x*)   : {result.fun:+.4f}')      # the cost at that input
print(f'  n evals : {result.nfev}')          # how many times J was called

**What you should see:** `success: True`, and a minimum at $x^* \approx -0.669$ with
$J(x^*) \approx -0.405$. Note that `result.x` is an **array** even for a one-variable problem, so
you index it with `[0]`.

---

## 3. Local vs Global: Why the Starting Point Matters <a id='3'></a>

The optimizer walks downhill from wherever it starts. If it starts on the wrong side of a hill, it
walks into the **nearer** valley and stops there, entirely unaware that a deeper one exists.

In [ ]:
# ---- Run the same optimizer from five different starting points ----
print(f'{"start x0":>10} {"found x*":>10} {"J(x*)":>10}')
for x0 in [-5.0, -1.0, 0.0, 1.0, 5.0]:
    r = minimize(J, x0)
    print(f'{x0:>10.1f} {r.x[0]:>10.3f} {r.fun:>10.4f}')

In [ ]:
# ---- Visualize why: mark both minima on the landscape ----
x = np.linspace(-4, 5, 500)
plt.figure()
plt.plot(x, J(x), lw=2, label='J(x)')
for x0, col in [(-5.0, 'tab:green'), (5.0, 'tab:red')]:
    r = minimize(J, x0)
    plt.plot(r.x[0], r.fun, 'o', ms=12, color=col, label=f'from x0={x0:+.0f}  ->  x*={r.x[0]:.2f}')
plt.xlabel('x'); plt.ylabel('J(x)'); plt.legend(); plt.title('The starting point decides the answer')
plt.show()

**What you should see:** Starting from $-5$, $-1$ or $0$ finds $x^* = -0.669$ with cost $-0.405$.
Starting from $+1$ or $+5$ finds $x^* = +1.860$ with cost $+0.232$ — a **worse** answer, but the
optimizer still reports `success: True`.

> **This is the single most important habit in this tutorial.** `success: True` means "I found a
> point where I cannot improve locally." It does **not** mean "this is the best answer." When a
> problem may have several minima, run it from several starting points and keep the best.

A cost function with only one minimum is called **convex**, and for convex problems the starting
point does not matter. The muscle cost in Part IV is convex, which is one of the reasons it was
chosen — but you should verify that rather than assume it.

---

## 4. Two Dimensions and Cost Landscapes <a id='4'></a>

With two decision variables the cost becomes a surface, and the optimizer rolls downhill on it.
The function now takes a **single array argument**, not two separate arguments — this is how SciPy
expects it, and getting it wrong is a common first error.

In [ ]:
# ---- A two-dimensional cost function ----
def J2(X):
    """X is an array [x, y]. Returns a single number."""
    x, y = X[0], X[1]
    return x * np.exp(-(x**2) - (y**2)) + ((x**2) + (y**2)) / 20

print(f'  J2([0, 0])   = {J2([0.0, 0.0]):+.4f}')
print(f'  J2([-0.7, 0]) = {J2([-0.7, 0.0]):+.4f}')

In [ ]:
# ---- Plot the cost surface as a contour map ----
xs = np.linspace(-3, 3, 200)
ys = np.linspace(-3, 3, 200)
XX, YY = np.meshgrid(xs, ys)
ZZ = XX * np.exp(-(XX**2) - (YY**2)) + ((XX**2) + (YY**2)) / 20

plt.figure(figsize=(6, 5))
c = plt.contourf(XX, YY, ZZ, levels=30, cmap=cm.viridis)
plt.colorbar(c, label='J(x, y)')
plt.contour(XX, YY, ZZ, levels=15, colors='w', linewidths=0.4)
plt.xlabel('x'); plt.ylabel('y'); plt.title('Two-dimensional cost landscape')
plt.gca().set_aspect('equal'); plt.grid(False)
plt.show()

In [ ]:
# ---- Minimize from two different starting points and mark them ----
plt.figure(figsize=(6, 5))
plt.contourf(XX, YY, ZZ, levels=30, cmap=cm.viridis)
plt.contour(XX, YY, ZZ, levels=15, colors='w', linewidths=0.4)
for x0, col in [([5, 5], 'tab:red'), ([1, 0], 'tab:orange')]:
    r = minimize(J2, x0)
    plt.plot(r.x[0], r.x[1], 'o', ms=12, color=col,
             label=f'from {x0}  ->  ({r.x[0]:.2f}, {r.x[1]:.2f}),  J={r.fun:.3f}')
    print(f'  start {str(x0):>8}  ->  x* = ({r.x[0]:+.3f}, {r.x[1]:+.3f}),  J = {r.fun:+.4f}')
plt.xlabel('x'); plt.ylabel('y'); plt.legend(fontsize=8, loc='upper left')
plt.title('Same lesson, two dimensions'); plt.gca().set_aspect('equal'); plt.grid(False)
plt.show()

**What you should see:** The dark basin near $(-0.67, 0)$ is the global minimum. Starting from
$[1, 0]$ lands in the shallow basin at $(+1.86, 0)$ instead. The two-dimensional case has exactly
the same trap as the one-dimensional case, but now it is harder to notice because you cannot plot
the landscape once you go past two variables — and the muscle problem has **six**.

---

# Part II — Adding Constraints

## 5. Bounds: Keeping Variables Non-Negative <a id='5'></a>

### Key concepts

Real decision variables usually cannot take any value. A muscle can **pull but never push**, so its
force must satisfy $F \ge 0$. In SciPy this is a **bound**, written as a list of
`(low, high)` pairs with `None` for no limit.

```python
bounds = [(0, None), (0, None)]   # both variables must be >= 0
```

In [ ]:
# ---- Without bounds, the optimizer is free to go negative ----
free = minimize(J, x0=0.0)
print(f'  unbounded : x* = {free.x[0]:+.4f}')

# ---- With a bound forcing x >= 0 ----
bounded = minimize(J, x0=0.0, bounds=[(0, None)])
print(f'  x >= 0    : x* = {bounded.x[0]:+.4f}   J = {bounded.fun:+.4f}')

**What you should see:** The unbounded answer is $-0.669$. Once $x \ge 0$ is imposed, that answer
is illegal and the optimizer returns $x^* = 0$ — pressed right up against the boundary. Solutions
sitting *exactly* on a bound are completely normal in constrained problems, and in Part IV most of
the muscles will end up at exactly zero force.

---

## 6. Equality Constraints <a id='6'></a>

An **equality constraint** says some function of the variables must equal zero. SciPy wants it as a
dictionary:

```python
{'type': 'eq', 'fun': lambda v: <expression that must be zero>}
```

To impose $x + y = 4$ you rewrite it as $x + y - 4 = 0$ and return the left-hand side. Everything
must be phrased as *"this expression equals zero"*.

In [ ]:
# ---- Minimize x^2 + y^2 subject to x + y = 4 ----
cost = lambda v: v[0]**2 + v[1]**2
con  = {'type': 'eq', 'fun': lambda v: v[0] + v[1] - 4}

r = minimize(cost, x0=[0.0, 0.0], constraints=[con])
print(f'  x* = {r.x}')
print(f'  check the constraint: x + y = {r.x.sum():.6f}  (should be 4)')
print(f'  cost at the optimum : {r.fun:.4f}')

**What you should see:** $(2, 2)$. Of all the pairs summing to 4, the one closest to the origin
splits the total evenly. That is not a coincidence — it is what a squared cost does, and it
previews the exponent lesson in Section 11.

> **Always verify the constraint yourself**, as the third line above does. An optimizer that fails
> to converge will still hand you an `x`, and it may violate the constraint silently.

---

## 7. SLSQP: The Workhorse <a id='7'></a>

SciPy picks an algorithm for you unless you name one. Once you have **both** bounds and equality
constraints, the method you want is **SLSQP** (Sequential Least SQuares Programming).

```python
minimize(cost, x0, method='SLSQP', bounds=..., constraints=[...])
```

### Useful options

| Option | What it does |
|---|---|
| `ftol` | convergence tolerance on the cost — tighten to `1e-12` for careful work |
| `maxiter` | iteration cap before it gives up |

In [ ]:
# ---- Same problem, now with both a constraint and bounds ----
r = minimize(cost, x0=[1.0, 1.0], method='SLSQP',
             bounds=[(0, None), (0, None)],
             constraints=[{'type': 'eq', 'fun': lambda v: v[0] + v[1] - 4}],
             options={'ftol': 1e-12, 'maxiter': 200})
print(f'  success : {r.success}   ({r.message})')
print(f'  x*      : {r.x}')
print(f'  residual: {abs(r.x.sum() - 4):.2e}')

---

## 8. Supplying Gradients <a id='8'></a>

By default SciPy estimates the slope of your cost by evaluating it at slightly different points —
**finite differences**. That works, but it is slow and slightly inaccurate. If you can write the
derivative by hand, pass it as `jac` and the optimizer becomes faster and more precise.

For $J = \sum_j x_j^2$ the derivative with respect to $x_j$ is $2x_j$, so `jac` returns the
**vector** $[2x_1, 2x_2, \dots]$ — one entry per decision variable.

In [ ]:
# ---- Compare: no gradient supplied vs. analytic gradient ----
cost = lambda v: np.sum(v**2)
grad = lambda v: 2*v                       # d/dx_j of sum(x^2) is 2*x_j
con  = [{'type': 'eq', 'fun': lambda v: v.sum() - 4}]

r_fd = minimize(cost, [1.0, 1.0], method='SLSQP', constraints=con)
r_an = minimize(cost, [1.0, 1.0], method='SLSQP', constraints=con, jac=grad)

print(f'  finite differences : {r_fd.nfev:3d} cost evaluations   x* = {r_fd.x}')
print(f'  analytic gradient  : {r_an.nfev:3d} cost evaluations   x* = {r_an.x}')

**What you should see:** Both find $(2, 2)$, but the analytic version calls the cost function far
fewer times. In Lab 06 you will solve this problem once per millisecond of movement, so the saving
is worth the two lines it takes to write the gradient.

---

# Part III — When There Is No Unique Answer

## 9. More Unknowns Than Equations <a id='9'></a>

### Key concepts

Consider a single equation in two unknowns:

$$x_1 + 2x_2 = 4, \qquad x_1, x_2 \ge 0$$

One equation, two unknowns. There is not one solution — there is a whole **line** of them. This is
an **underdetermined** system, and it is the mathematical shape of the muscle redundancy problem.

In [ ]:
# ---- Draw the full set of valid solutions ----
x1 = np.linspace(0, 4, 200)
x2 = (4 - x1) / 2
keep = x2 >= 0

plt.figure(figsize=(5.5, 5))
plt.plot(x1[keep], x2[keep], lw=3, label='every point here satisfies the equation')
for pt in [[0, 2], [2, 1], [4, 0]]:
    plt.plot(pt[0], pt[1], 'o', ms=10)
    plt.annotate(f'({pt[0]}, {pt[1]})', pt, xytext=(6, 6), textcoords='offset points', fontsize=9)
plt.xlabel('$x_1$'); plt.ylabel('$x_2$'); plt.legend(fontsize=9)
plt.title('One equation, two unknowns: a line of solutions')
plt.gca().set_aspect('equal'); plt.show()

for pt in [[0, 2], [2, 1], [4, 0]]:
    print(f'  x = {pt}   ->   x1 + 2*x2 = {pt[0] + 2*pt[1]}')

**What you should see:** Every point on that line is a perfectly valid answer. The mathematics
cannot prefer one over another — **nothing in the equation itself picks a solution.**

> The number of free directions is `n_unknowns - rank(A)`. Here $2 - 1 = 1$, which is why the
> solution set is a line. In Part IV it will be $6 - 2 = 4$, a four-dimensional set you cannot draw.

---

## 10. Using a Cost to Choose <a id='10'></a>

To get a unique answer you must add something the equations do not contain: a cost. Different costs
select different points on the same line.

In [ ]:
# ---- Solve the same underdetermined system under three different costs ----
A = np.array([[1.0, 2.0]])
b = np.array([4.0])
con = [{'type': 'eq', 'fun': lambda v: A @ v - b}]

solutions = {}
for name, f in [('sum of |x|   (n=1)', lambda v: np.sum(v)),
                ('sum of x^2   (n=2)', lambda v: np.sum(v**2)),
                ('sum of x^3   (n=3)', lambda v: np.sum(v**3))]:
    r = minimize(f, [1.0, 1.0], method='SLSQP', bounds=[(0, None)]*2,
                 constraints=con, options={'ftol': 1e-12})
    solutions[name] = r.x
    print(f'  {name} ->  x = {r.x}   (check: {(A @ r.x)[0]:.4f})')

In [ ]:
# ---- Show where each cost lands on the line of solutions ----
plt.figure(figsize=(5.5, 5))
plt.plot(x1[keep], x2[keep], lw=3, color='lightgray', label='all valid solutions')
for (name, sol), col in zip(solutions.items(), ['tab:red', 'tab:blue', 'tab:green']):
    plt.plot(sol[0], sol[1], 'o', ms=12, color=col, label=name)
plt.xlabel('$x_1$'); plt.ylabel('$x_2$'); plt.legend(fontsize=9)
plt.title('The cost function decides which solution you get')
plt.gca().set_aspect('equal'); plt.show()

**What you should see:** The linear cost ($n=1$) picks the **corner** at $(0, 2)$ — it uses as few
variables as possible. The squared cost picks $(0.8, 1.6)$ and the cubic $(1.05, 1.48)$, both
spreading the work across both variables.

> **This is the central idea of the whole tutorial.** When a problem is underdetermined, the answer
> you get is a property of the cost you chose, not of the physics. Reporting a solution without
> stating the cost is reporting half the result.

---

## 11. What the Exponent Does <a id='11'></a>

The pattern above generalizes. A cost $J = \sum_j x_j^n$ with larger $n$ penalizes large values
disproportionately, so it prefers to spread effort out rather than concentrate it.

In [ ]:
# ---- Sweep the exponent and watch the solution move ----
print(f'{"n":>5} {"x1":>8} {"x2":>8} {"largest":>9} {"total":>8}')
ns = [1, 1.5, 2, 3, 4, 6, 10]
xs_n, largest, totals = [], [], []
for n in ns:
    r = minimize(lambda v: np.sum(v**n), [1.0, 1.0], method='SLSQP',
                 bounds=[(0, None)]*2, constraints=con, options={'ftol': 1e-12})
    xs_n.append(r.x); largest.append(r.x.max()); totals.append(r.x.sum())
    print(f'{n:>5} {r.x[0]:>8.3f} {r.x[1]:>8.3f} {r.x.max():>9.3f} {r.x.sum():>8.3f}')

In [ ]:
# ---- Plot the trade-off ----
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ns, largest, 'o-', lw=2, color='tab:red')
ax[0].set_xlabel('cost exponent n'); ax[0].set_ylabel('largest single variable')
ax[0].set_title('Higher n lowers the peak')
ax[1].plot(ns, totals, 's-', lw=2, color='tab:blue')
ax[1].set_xlabel('cost exponent n'); ax[1].set_ylabel('sum of all variables')
ax[1].set_title('...by raising the total')
plt.tight_layout(); plt.show()

**What you should see:** As $n$ rises the largest variable falls while the total rises. The
exponent is buying a lower peak by paying more in total.

That trade is exactly the argument Crowninshield & Brand made for muscle: a muscle's endurance
depends on its force relative to its capacity, so a nervous system trying to delay fatigue should
keep the *peak* stress down even at the price of activating more tissue overall. They settled on
$n = 3$.

---

# Part IV — The Muscle Redundancy Problem

## 12. Six Muscles, Two Torques <a id='12'></a>

You now have every piece. The arm in this course has two joints and six muscles. Each muscle $j$
pulls with force $F_j \ge 0$ and produces joint torque through its **moment arms**, collected in a
$2 \times 6$ matrix $R$:

$$\tau = R\,F, \qquad F \ge 0$$

Two equations, six unknowns — the same shape as Section 9, four dimensions bigger. The cost is the
Crowninshield criterion, where $\rho_j$ is muscle $j$'s force capacity:

$$J = \sum_{j=1}^{6}\left(\frac{F_j}{\rho_j}\right)^{n}$$

In [ ]:
# ---- The arm's moment arms and muscle capacities ----
NAMES = ['pec', 'bic_l', 'bic_s', 'delt', 'tri_l', 'tri_lg']
R = np.array([[ 4.0, 0.0, 2.5, -4.0,  0.0, -4.0],      # shoulder moment arms (cm)
              [ 0.0, 3.0, 3.0,  0.0, -2.0, -2.0]])     # elbow moment arms (cm)
rho = np.array([14.9, 11.0, 2.1, 14.9, 12.1, 6.7])     # force capacity (N)

print('R shape:', R.shape, '  rank:', np.linalg.matrix_rank(R))
print('free directions (unknowns - rank):', 6 - np.linalg.matrix_rank(R))

In [ ]:
# ---- Set up and solve the muscle problem ----
tau = np.array([-160.0, -20.0])            # required torque (N.cm)
n_pow = 3

def muscle_cost(F):     return np.sum((F / rho) ** n_pow)
def muscle_grad(F):     return n_pow * (F / rho) ** (n_pow - 1) / rho

res = minimize(muscle_cost, np.ones(6), jac=muscle_grad, method='SLSQP',
               bounds=[(0, None)] * 6,
               constraints=[{'type': 'eq', 'fun': lambda F: R @ F - tau,
                             'jac': lambda F: R}],
               options={'ftol': 1e-12, 'maxiter': 400})

F = np.maximum(res.x, 0)
for j, nm in enumerate(NAMES):
    print(f'  {nm:7s} {F[j]:7.2f} N')
print(f'\n  required tau : {tau}')
print(f'  R @ F        : {R @ F}')
print(f'  residual     : {np.linalg.norm(R @ F - tau):.2e}')

**What you should see:** Only two or three muscles carry any force at all; the rest are exactly
zero. The residual should be around $10^{-14}$ — the constraint is satisfied to machine precision.

---

## 13. Reading the Solution Critically <a id='13'></a>

The optimizer returned an answer. Before believing it, ask three questions.

In [ ]:
# ---- Question 1: is the constraint actually satisfied? ----
assert np.allclose(R @ F, tau, atol=1e-8), 'constraint violated'
print('1. Constraint satisfied.')

# ---- Question 2: are the bounds respected? ----
assert (F >= -1e-9).all(), 'negative muscle force'
print('2. All forces non-negative.')

# ---- Question 3: is this a local minimum only? Try many starting points. ----
rng = np.random.default_rng(0)
best_cost, worst_gap = muscle_cost(F), 0.0
for _ in range(20):
    r = minimize(muscle_cost, rng.uniform(0, 30, 6), jac=muscle_grad, method='SLSQP',
                 bounds=[(0, None)]*6,
                 constraints=[{'type': 'eq', 'fun': lambda Fv: R @ Fv - tau, 'jac': lambda Fv: R}],
                 options={'ftol': 1e-12, 'maxiter': 400})
    if r.success:
        worst_gap = max(worst_gap, abs(muscle_cost(np.maximum(r.x, 0)) - best_cost))
print(f'3. Largest cost difference over 20 random starts: {worst_gap:.2e}')

**What you should see:** All three pass, and the third returns a tiny number — every starting
point reaches the same cost. That tells you this particular problem is **convex**, so unlike
Section 3 the starting point does not matter here.

That is a fact you *checked*, not one you assumed. Convexity comes from the cost being a sum of
increasing powers over a set defined by linear constraints. Change the cost to something
non-convex and the Section 3 problem returns immediately.

In [ ]:
# ---- Why are most muscles at zero? Look at the geometry. ----
plt.figure(figsize=(6, 5.5))
for j, nm in enumerate(NAMES):
    plt.arrow(0, 0, R[0, j], R[1, j], head_width=0.15, lw=2,
              length_includes_head=True, color=plt.cm.tab10(j))
    plt.annotate(nm, (R[0, j]*1.12, R[1, j]*1.12), fontsize=9, ha='center')
u = tau / np.linalg.norm(tau) * 4
plt.arrow(0, 0, u[0], u[1], head_width=0.22, lw=3, color='k', length_includes_head=True)
plt.annotate('required torque', (u[0]*1.05, u[1]*1.05 - 0.5), fontsize=10, fontweight='bold')
plt.xlim(-6, 6); plt.ylim(-5, 5); plt.gca().set_aspect('equal')
plt.xlabel('shoulder moment arm (cm)'); plt.ylabel('elbow moment arm (cm)')
plt.title('Only muscles pointing the same way as tau can help')
plt.show()

**What you should see:** Each arrow is one column of $R$ — the torque that muscle produces per
newton of force. The black arrow is the direction the torque must point. Muscles pointing the
*opposite* way cannot contribute, because their force cannot go negative to flip them around.

So the "four free directions" from Section 12 overstate the real freedom badly. Non-negativity
removes most of it before the cost function ever gets a say.

---

# Part V — Practice

## 14. Exercises <a id='14'></a>

---

### Exercise 1: Find Both Minima

Write a loop that runs `minimize(J, x0)` from 50 starting points spanning $-10$ to $+10$, collects
every distinct minimum it finds (round to 3 decimals), and prints them sorted by cost. This is the
**multi-start** strategy, and it is what you do whenever you are unsure a problem is convex.

In [ ]:
# ---- YOUR CODE HERE ----

### Exercise 2: A Constraint You Design

Minimize $J = x^2 + y^2$ subject to $x - y = 2$, with both variables bounded to $[-5, 5]$.
Predict the answer before you run it, then check whether the optimizer agrees.

In [ ]:
# ---- YOUR CODE HERE ----

### Exercise 3: The Exponent and the Muscle Solution

Using the `R`, `rho` and `tau` from Section 12, solve the muscle problem for $n = 1, 2, 3, 5$ and
print the force vector each time. Then answer, in a comment:

1. How many muscles are active at $n = 1$, and how many at $n = 5$?
2. Which $n$ gives the largest single muscle force?
3. Section 11 said higher $n$ spreads effort out. Does the muscle problem follow that pattern?

In [ ]:
# ---- YOUR CODE HERE ----

### Exercise 4: A Torque the Arm Cannot Produce (Challenge)

Try to find a torque $\tau$ for which the optimizer **fails** — one no combination of non-negative
muscle forces can produce. Use the arrow plot from Section 13 to reason about which direction to
pick, then confirm by checking `res.success` and the constraint residual.

*Hint: all six arrows must lie on one side of some line through the origin for a direction to be
unreachable. Look at where the arrows are sparse.*

In [ ]:
# ---- YOUR CODE HERE ----

---

## 15. Summary & Further Reading <a id='15'></a>

### What You Learned

| Concept | Key point |
|---|---|
| Cost function | An assumption you supply, not something the physics gives you |
| `minimize(fun, x0)` | Walks downhill from `x0`; `success: True` means *locally* stuck |
| Local vs global | Multi-start whenever convexity is not established |
| Bounds | `[(0, None)] * n` for non-negative variables; solutions often sit *on* a bound |
| Equality constraints | `{'type': 'eq', 'fun': ...}`, phrased as *expression = 0* |
| SLSQP | The method to use once you have both bounds and constraints |
| `jac` | Analytic gradients: faster and more accurate, worth writing |
| Underdetermined | `n_unknowns - rank(A)` free directions; the cost picks among them |
| Exponent $n$ | Higher $n$ lowers the peak by raising the total |
| Non-negativity | Removes far more freedom than the null-space dimension suggests |

### The Static Optimization Recipe

```python
cost = lambda v: ...                                    # 1. what to minimize
grad = lambda v: ...                                    # 2. its gradient (optional but worth it)
cons = [{'type': 'eq', 'fun': lambda v: A @ v - b,      # 3. what must hold exactly
         'jac': lambda v: A}]
bnds = [(0, None)] * n                                  # 4. what range each variable may take

res = minimize(cost, x0, jac=grad, method='SLSQP',      # 5. solve
               bounds=bnds, constraints=cons,
               options={'ftol': 1e-12, 'maxiter': 400})

assert res.success                                      # 6. ALWAYS check
assert np.allclose(A @ res.x, b)                        #    the constraint, yourself
```

### Common Pitfalls

| Symptom | Likely cause |
|---|---|
| Answer changes with `x0` | Non-convex cost — use multi-start |
| `success: True` but constraint violated | You never checked; add the assert |
| Optimizer very slow | No `jac` supplied, or `ftol` far too tight |
| Negative values in the answer | Bounds not passed, or method does not support them |
| `success: False` | The constraints may be infeasible — see Exercise 4 |

### Further Reading

- Crowninshield, R. D., & Brand, R. A. (1981). A physiologically based criterion of muscle force
  prediction in locomotion. *Journal of Biomechanics*, **14**(11), 793–801.
- SciPy documentation: [`scipy.optimize.minimize`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)
- Erdemir, A., et al. (2007). Model-based estimation of muscle forces exerted during movements.
  *Clinical Biomechanics*, **22**(2), 131–154. — a review of how these methods are used in practice.

---

**Next:** Lab 06 uses everything here, solving this problem once per millisecond of a reaching
movement, and then asks what the resulting muscle forces mean.